# Introduction to Recommender Systems

<p align="center">
    <img width="721" alt="cover-image" src="https://user-images.githubusercontent.com/49638680/204351915-373011d3-75ac-4e21-a6df-99cd1c552f2c.png">
</p>

---

# The Exercise

In this exercise, you will implement a simple recommender system using the GoodRead dataset. The dataset contains information about books, authors, and user ratings. The goal is to recommend books to users based on their ratings.

## The Dataset

The dataset is available at this link: [GoodReads Dataset](https://github.com/zygmuntz/goodbooks-10k/releases/download/v1.0/goodbooks-10k.zip). It contains the following files:

- `books.csv`: Contains information about books.
- `ratings.csv`: Contains user ratings for books.
- `book_tags.csv`: Contains tags assigned to books.

## The Task

You need to implement a simple recommender system that recommends books to users based on their ratings.

### Steps

1. Download the dataset.
2. Perfom data analysis.
3. Justify all the choices you make as results of some data analysis.
4. Implement the recommender system.

### Tasks

While implementing the recommender system, you need to perform the following tasks (that are also a guideline for the implementation).

#### Data Analysis

- Load and explore the dataset.
- Analyse the distribution of ratings.
- Visualise the number of ratings per book and per user.
- Check for missing values and decide how to handle them.


#### Feature Engineering

- Encode User_ID and Book_Title as integer indices for the model.
- Create a feature to represent the popularity of books based on the number of ratings.
- Optionally, incorporate author information if it improves the model.

#### Building the Recommender System

- Split the data into training and test sets.
- Implement a collaborative filtering model using matrix factorization.
- Train the model and evaluate its performance using RMSE (Root Mean Square Error) on the test set.
- Generate book recommendations for a set of users.- 

#### Recommendations

- Write a function to display the top N recommended books for a given user.
- Discuss how different features and the size of the dataset affect the model's performance.

In [ ]:
# Imports et réglages généraux
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)
%matplotlib inline


## 1. Téléchargement et chargement des données

La cellule suivante cherche d'abord le dossier de données à plusieurs emplacements. Le téléchargement n'est effectué que si les fichiers ne sont pas déjà présents. Cela permet d'exécuter le notebook aussi bien depuis sa propre arborescence que depuis la racine du projet.

In [ ]:
DATA_URL = (
    "https://github.com/zygmuntz/goodbooks-10k/releases/download/"
    "v1.0/goodbooks-10k.zip"
)
candidate_dirs = [
    Path("data/goodbooks-10k"),
    Path("../data/goodbooks-10k"),
    Path("../../data/goodbooks-10k"),
]
data_dir = next(
    (path for path in candidate_dirs if (path / "ratings.csv").exists()),
    candidate_dirs[0],
)

if not (data_dir / "ratings.csv").exists():
    data_dir.mkdir(parents=True, exist_ok=True)
    archive_path = data_dir / "goodbooks-10k.zip"
    print("Téléchargement du jeu de données…")
    urlretrieve(DATA_URL, archive_path)
    with ZipFile(archive_path) as archive:
        archive.extractall(data_dir)
    archive_path.unlink()

ratings = pd.read_csv(
    data_dir / "ratings.csv",
    dtype={"user_id": "int32", "book_id": "int32", "rating": "int8"},
)
books = pd.read_csv(data_dir / "books.csv")

print(f"Dossier utilisé : {data_dir.resolve()}")
print(f"Notes : {ratings.shape[0]:,} lignes et {ratings.shape[1]} colonnes")
print(f"Livres : {books.shape[0]:,} lignes et {books.shape[1]} colonnes")
display(ratings.head(), books[["book_id", "title", "authors"]].head())


## 2. Analyse exploratoire

Je vérifie d'abord la qualité des données. Les colonnes nécessaires au modèle sont les identifiants utilisateur/livre et la note. Pour afficher les recommandations, il faut aussi le titre et l'auteur.

In [ ]:
quality_report = pd.DataFrame({
    "type": ratings.dtypes.astype(str),
    "valeurs_manquantes": ratings.isna().sum(),
    "valeurs_uniques": ratings.nunique(),
})
display(quality_report)

relevant_book_columns = ["book_id", "title", "authors"]
print("Valeurs manquantes dans les métadonnées utiles :")
display(books[relevant_book_columns].isna().sum().to_frame("nombre"))
print("Doublons (même utilisateur, même livre) :",
      ratings.duplicated(["user_id", "book_id"]).sum())
print("Note minimale / maximale :", ratings["rating"].min(),
      "/", ratings["rating"].max())


**Conclusion sur les valeurs manquantes.** Les trois colonnes de `ratings` sont complètes et il n'y a pas de doublon utilisateur-livre. Il n'est donc pas nécessaire de supprimer ou d'inventer des notes. Les valeurs absentes des autres colonnes de `books.csv` ne gênent pas le modèle, car seules les métadonnées complètes `title` et `authors` seront affichées.

In [ ]:
rating_counts = ratings["rating"].value_counts().sort_index()
rating_percentages = 100 * rating_counts / rating_counts.sum()
display(pd.DataFrame({"nombre": rating_counts, "pourcentage": rating_percentages.round(2)}))

ax = sns.countplot(data=ratings, x="rating", color="steelblue")
ax.set(title="Distribution des notes", xlabel="Note", ylabel="Nombre d'évaluations")
plt.show()

print(f"Note moyenne : {ratings['rating'].mean():.3f}")
print(f"Part des notes 4 ou 5 : {(ratings['rating'].ge(4).mean() * 100):.2f} %")


La distribution est déséquilibrée vers les bonnes notes : environ 69 % des évaluations valent 4 ou 5, et la moyenne est proche de 3,92. Le RMSE reste utile, mais il faut le comparer à un modèle naïf qui prédit toujours la moyenne. Sinon un score peut sembler bon simplement parce que les notes sont souvent élevées.

In [ ]:
ratings_per_book = ratings.groupby("book_id").size().sort_values(ascending=False)
ratings_per_user = ratings.groupby("user_id").size().sort_values(ascending=False)

display(pd.DataFrame({
    "notes_par_livre": ratings_per_book.describe(),
    "notes_par_utilisateur": ratings_per_user.describe(),
}).round(2))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(np.arange(1, len(ratings_per_book) + 1), ratings_per_book.values)
axes[0].set(yscale="log", title="Nombre de notes par livre (échelle logarithmique)",
            xlabel="Rang du livre", ylabel="Nombre de notes")
axes[1].hist(ratings_per_user.values, bins=30, color="darkorange", edgecolor="white")
axes[1].set(title="Nombre de notes par utilisateur",
            xlabel="Nombre de notes", ylabel="Nombre d'utilisateurs")
plt.tight_layout()
plt.show()


On observe une vraie **longue traîne pour les livres** : quelques titres ont énormément de notes, alors que la moitié des livres en ont au plus environ 248. L'activité des utilisateurs est beaucoup plus régulière (médiane proche de 111 notes). Je conserve toutes les interactions : même les livres moins connus contiennent une information utile pour personnaliser les recommandations. La popularité servira surtout de solution de repli pour un nouvel utilisateur.

## 3. Création des variables

Les identifiants d'origine sont transformés en indices consécutifs, indispensables pour indexer les matrices de facteurs. Un `book_id` correspond à un titre unique dans `books.csv` : encoder le livre revient donc bien à encoder son titre, tout en évitant de répéter de longues chaînes de caractères dans les 6 millions de lignes.

In [ ]:
user_ids = np.sort(ratings["user_id"].unique())
book_ids = np.sort(ratings["book_id"].unique())
user_to_idx = pd.Series(np.arange(len(user_ids), dtype=np.int32), index=user_ids)
book_to_idx = pd.Series(np.arange(len(book_ids), dtype=np.int32), index=book_ids)
idx_to_user = pd.Series(user_ids, index=np.arange(len(user_ids)))
idx_to_book = pd.Series(book_ids, index=np.arange(len(book_ids)))

ratings["user_idx"] = ratings["user_id"].map(user_to_idx).astype("int32")
ratings["book_idx"] = ratings["book_id"].map(book_to_idx).astype("int32")
books["book_idx"] = books["book_id"].map(book_to_idx)

print(f"{len(user_ids):,} utilisateurs encodés de 0 à {len(user_ids)-1:,}")
print(f"{len(book_ids):,} livres encodés de 0 à {len(book_ids)-1:,}")
display(books[["book_id", "book_idx", "title", "authors"]].head())


In [ ]:
# La popularité est calculée sans utiliser la future partie test.
train, test = train_test_split(
    ratings, test_size=0.20, random_state=RANDOM_STATE,
)

book_features = train.groupby("book_id")["rating"].agg(
    rating_count="count", mean_rating="mean"
)
book_features["popularity"] = (
    np.log1p(book_features["rating_count"])
    / np.log1p(book_features["rating_count"].max())
)

# Score bayésien : il évite qu'un livre avec très peu de notes soit premier par hasard.
global_train_mean = train["rating"].mean()
minimum_votes = book_features["rating_count"].quantile(0.75)
v = book_features["rating_count"]
R = book_features["mean_rating"]
book_features["weighted_rating"] = (
    (v / (v + minimum_votes)) * R
    + (minimum_votes / (v + minimum_votes)) * global_train_mean
)
display(book_features.sort_values("popularity", ascending=False).head())
print(f"Apprentissage : {len(train):,} notes | Test : {len(test):,} notes")


Le logarithme réduit l'écart énorme entre les livres très connus et les autres. Le score bayésien combine quantité et qualité : la moyenne d'un livre peu évalué est rapprochée de la moyenne globale. Je n'utilise pas directement l'auteur dans cette première version, car la consigne principale porte sur le filtrage collaboratif et les auteurs forment une variable textuelle très dispersée. Elle serait surtout utile dans un modèle hybride ou pour le démarrage à froid.

## 4. Factorisation de matrice

Une matrice dense de taille 53 424 × 10 000 demanderait plusieurs gigaoctets. Je construis donc une matrice sparse qui ne stocke que les notes observées. Le modèle prédit : moyenne globale + biais utilisateur + biais livre + produit des facteurs latents. Les biais capturent les utilisateurs généreux et les livres généralement appréciés ; la SVD des résidus apprend ensuite les goûts plus fins.

In [ ]:
def fit_biases(frame, n_users, n_books, regularization=10.0, iterations=5):
    """Apprend des biais utilisateur et livre régularisés."""
    user_idx = frame["user_idx"].to_numpy()
    book_idx = frame["book_idx"].to_numpy()
    values = frame["rating"].to_numpy(dtype=np.float32)
    mean = float(values.mean())
    user_bias = np.zeros(n_users, dtype=np.float32)
    book_bias = np.zeros(n_books, dtype=np.float32)
    user_count = np.bincount(user_idx, minlength=n_users)
    book_count = np.bincount(book_idx, minlength=n_books)

    for _ in range(iterations):
        user_bias = np.bincount(
            user_idx, weights=values - mean - book_bias[book_idx], minlength=n_users
        ) / (user_count + regularization)
        book_bias = np.bincount(
            book_idx, weights=values - mean - user_bias[user_idx], minlength=n_books
        ) / (book_count + regularization)

    return mean, user_bias.astype(np.float32), book_bias.astype(np.float32)


def fit_matrix_factorization(frame, n_users, n_books, n_factors=20):
    """Factorise la matrice sparse des résidus par SVD tronquée."""
    mean, user_bias, book_bias = fit_biases(frame, n_users, n_books)
    user_idx = frame["user_idx"].to_numpy()
    book_idx = frame["book_idx"].to_numpy()
    values = frame["rating"].to_numpy(dtype=np.float32)
    residuals = values - mean - user_bias[user_idx] - book_bias[book_idx]
    residual_matrix = csr_matrix(
        (residuals, (user_idx, book_idx)), shape=(n_users, n_books)
    )

    U, singular_values, Vt = svds(
        residual_matrix, k=n_factors, random_state=RANDOM_STATE
    )
    order = np.argsort(singular_values)[::-1]
    singular_values = singular_values[order]
    U, Vt = U[:, order], Vt[order, :]
    sqrt_s = np.sqrt(singular_values)
    user_factors = (U * sqrt_s).astype(np.float32)
    book_factors = (Vt.T * sqrt_s).astype(np.float32)
    return mean, user_bias, book_bias, user_factors, book_factors


n_users, n_books = len(user_ids), len(book_ids)
global_mean, user_bias, book_bias, user_factors, book_factors = (
    fit_matrix_factorization(train, n_users, n_books, n_factors=20)
)
print("Facteurs utilisateurs :", user_factors.shape)
print("Facteurs livres :", book_factors.shape)


In [ ]:
def predict_pairs(frame):
    """Prédit les notes des couples utilisateur-livre d'un DataFrame."""
    user_idx = frame["user_idx"].to_numpy()
    book_idx = frame["book_idx"].to_numpy()
    latent_part = np.sum(
        user_factors[user_idx] * book_factors[book_idx], axis=1
    )
    predictions = (
        global_mean + user_bias[user_idx] + book_bias[book_idx] + latent_part
    )
    return np.clip(predictions, 1.0, 5.0)


test_predictions = predict_pairs(test)
true_ratings = test["rating"].to_numpy(dtype=np.float32)
model_rmse = float(np.sqrt(np.mean((test_predictions - true_ratings) ** 2)))
baseline_predictions = np.full(len(test), global_mean, dtype=np.float32)
baseline_rmse = float(np.sqrt(np.mean((baseline_predictions - true_ratings) ** 2)))

results = pd.DataFrame({
    "modèle": ["Moyenne globale (référence)", "Biais + factorisation SVD"],
    "RMSE": [baseline_rmse, model_rmse],
})
display(results.style.format({"RMSE": "{:.4f}"}))
print(f"Amélioration du RMSE : {(1 - model_rmse / baseline_rmse) * 100:.2f} %")


Le score est calculé uniquement sur les 20 % de notes mises de côté. Un RMSE inférieur à celui de la moyenne globale montre que le modèle apprend bien une information personnalisée. Une erreur d'environ 0,84 signifie que la prédiction s'écarte en moyenne quadratique de moins d'un point sur l'échelle de 1 à 5. Ce n'est pas parfait : une note dépend aussi du contexte et la SVD ne connaît ni le texte du livre ni son auteur.

## 5. Génération de recommandations

Pour un utilisateur connu, la fonction classe les livres non encore notés selon la note prédite. Pour un identifiant inconnu, il n'existe aucun facteur utilisateur : on retourne alors les meilleurs livres selon le score bayésien de popularité.

In [ ]:
book_catalog = (
    books[["book_id", "book_idx", "title", "authors"]]
    .dropna(subset=["book_idx"])
    .assign(book_idx=lambda frame: frame["book_idx"].astype("int32"))
    .set_index("book_idx")
    .sort_index()
)


def recommend_books(user_id, top_n=10):
    """Affiche les top_n livres non vus pour un utilisateur."""
    if top_n <= 0:
        return pd.DataFrame(columns=["title", "authors", "predicted_rating"])

    if user_id not in user_to_idx.index:
        popular = (
            book_features.join(books.set_index("book_id")[["title", "authors"]])
            .sort_values(["weighted_rating", "rating_count"], ascending=False)
            .head(top_n)
        )
        print("Nouvel utilisateur : recommandations populaires et bien notées.")
        return popular[["title", "authors", "weighted_rating", "rating_count"]]

    user_idx = int(user_to_idx.loc[user_id])
    scores = (
        global_mean + user_bias[user_idx] + book_bias
        + book_factors @ user_factors[user_idx]
    )
    seen_book_ids = ratings.loc[ratings["user_id"] == user_id, "book_id"]
    seen_indices = book_to_idx.loc[seen_book_ids].to_numpy(dtype=np.int32)
    scores[seen_indices] = -np.inf

    available = np.isfinite(scores)
    candidate_indices = np.flatnonzero(available)
    n = min(top_n, len(candidate_indices))
    best_local = np.argpartition(scores[candidate_indices], -n)[-n:]
    best_indices = candidate_indices[best_local]
    best_indices = best_indices[np.argsort(scores[best_indices])[::-1]]

    result = book_catalog.loc[best_indices, ["book_id", "title", "authors"]].copy()
    result["predicted_rating"] = np.clip(scores[best_indices], 1.0, 5.0)
    return result.reset_index(drop=True)


display(recommend_books(user_id=1, top_n=10))
display(recommend_books(user_id=999_999, top_n=5))


## 6. Vérifications automatiques

In [ ]:
# Ces tests vérifient les points les plus importants du notebook.
assert ratings[["user_id", "book_id", "rating"]].isna().sum().sum() == 0
assert ratings.duplicated(["user_id", "book_id"]).sum() == 0
assert user_factors.shape == (n_users, 20)
assert book_factors.shape == (n_books, 20)
assert np.all((test_predictions >= 1) & (test_predictions <= 5))
assert model_rmse < baseline_rmse

sample_user = int(user_ids[0])
sample_recommendations = recommend_books(sample_user, top_n=10)
already_seen = set(ratings.loc[ratings["user_id"] == sample_user, "book_id"])
assert len(sample_recommendations) == 10
assert already_seen.isdisjoint(sample_recommendations["book_id"])
assert sample_recommendations["predicted_rating"].is_monotonic_decreasing

print("Tous les tests sont passés avec succès.")


## 7. Discussion finale

- **Taille des données :** davantage de notes aide à apprendre les goûts, surtout pour les livres peu connus, mais augmente le temps et la mémoire nécessaires. La matrice sparse permet de conserver les 6 millions de notes sans créer une énorme matrice dense.
- **Nombre de facteurs :** avec trop peu de facteurs, le modèle simplifie excessivement les goûts ; avec trop de facteurs, il devient plus lent et risque d'apprendre du bruit. Vingt facteurs donnent ici un compromis simple et rapide.
- **Popularité :** elle est utile pour les nouveaux utilisateurs, mais l'utiliser seule produirait les mêmes recommandations pour tout le monde et renforcerait la longue traîne.
- **Auteur et contenu :** ajouter les auteurs, genres ou tags pourrait améliorer les recommandations pour les nouveaux livres et expliquer plus facilement les résultats. Ce serait une extension hybride du modèle actuel.
- **Limite du RMSE :** il mesure correctement l'erreur sur les notes, mais pas directement la qualité d'une liste de recommandations. Pour aller plus loin, il faudrait aussi calculer Precision@K, Recall@K, diversité et couverture du catalogue.